In [ ]:
!pip install xgboost


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Load dataset
data = pd.read_csv("/Users/harinathrayala/Desktop/health_dataset.csv")

# Separate features and target
X = data.drop('HeartDisease', axis=1)
y = data['HeartDisease']

# Standardize numeric columns
num_cols = ['Age', 'BMI', 'BloodPressure', 'Cholesterol', 'Glucose']
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# XGBoost model with optimized hyperparameters
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    reg_lambda=1.5,
    reg_alpha=0.2,
    scale_pos_weight=1,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# Cross-validation for stability
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Train model
xgb_model.fit(X_train, y_train)

# Predict
y_pred = xgb_model.predict(X_test)

# Evaluation
acc = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {acc*100:.2f}%\n")
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Feature Importance
import matplotlib.pyplot as plt
import xgboost as xgb
xgb.plot_importance(xgb_model)
plt.title("Feature Importance")
plt.show()
